# Week 3 — ID3 decision trees
Machine Learning Club · Laboratory

This notebook implements categorical ID3. The linked interactive webapp visualises CART; it is related but not the same algorithm.

Goal: calculate entropy, select a root, build a tree and inspect an unseen case.

## 1. Data
Three categorical features (sky, wind, temperature), one label (go). Every possible combination occurs once. Labels are deliberately synthetic.

In [ ]:
from collections import Counter
from math import log2
examples = [
    {"sky":"sunny","wind":"calm","temperature":"warm","go":"yes"},
    {"sky":"sunny","wind":"calm","temperature":"cold","go":"yes"},
    {"sky":"sunny","wind":"windy","temperature":"warm","go":"yes"},
    {"sky":"sunny","wind":"windy","temperature":"cold","go":"no"},
    {"sky":"rainy","wind":"calm","temperature":"warm","go":"no"},
    {"sky":"rainy","wind":"calm","temperature":"cold","go":"no"},
    {"sky":"rainy","wind":"windy","temperature":"warm","go":"no"},
    {"sky":"rainy","wind":"windy","temperature":"cold","go":"no"},
]
features = ["sky","wind","temperature"]
for i,row in enumerate(examples,1):print(i,row)

## 2. Entropy and information gain
Shannon entropy is H(Y) = -sum(p_i * log2(p_i)). The definition extends continuously to probability zero. Information gain subtracts the weighted entropy after the split.

In [ ]:
def entropy(rows):
    if not rows:return 0.0
    counts = Counter(r["go"] for r in rows)
    total = len(rows)
    return -sum((v/total)*log2(v/total) for v in counts.values())

def gain(rows, feature):
    parent=entropy(rows)
    remainder=0.0
    for value in sorted({r[feature] for r in rows}):
        subset=[r for r in rows if r[feature]==value]
        remainder+=len(subset)/len(rows)*entropy(subset)
    return parent-remainder

print("Parent entropy",round(entropy(examples),4))
for f in features:print(f,"gain",round(gain(examples,f),4))

## 3. Construct categorical ID3
At each impure node choose the remaining attribute with the largest information gain, and recurse on every observed category. If there are no remaining attributes, use the majority class. Ties use the original feature order.

In [ ]:
def majority(rows):
    return Counter(r["go"] for r in rows).most_common(1)[0][0]

def id3(rows, available):
    labels={r["go"] for r in rows}
    if len(labels)==1:return {"label":rows[0]["go"]}
    if not available:return {"label":majority(rows)}
    feature=max(available,key=lambda f:gain(rows,f))
    options={}
    for value in sorted({r[feature] for r in rows}):
        subset=[r for r in rows if r[feature]==value]
        options[value]=id3(subset,[f for f in available if f!=feature])
    return {"test":feature,"default":majority(rows),"branches":options}

def classify(tree,sample):
    while "label" not in tree:
        value=sample.get(tree["test"])
        tree=tree["branches"].get(value,{"label":tree["default"]})
    return tree["label"]

tree=id3(examples,features)
print("Tree",tree)
print("Training correct",sum(classify(tree,r)==r["go"] for r in examples),"/",len(examples))

## 4. Experiment
Predict the three information gains before running. Why does the rainy branch terminate? Remove the sunny/windy/cold/no record, fit again, and predict the missing case. Does fitting all remaining examples guarantee a correct new prediction?

In [ ]:
heldout={"sky":"sunny","wind":"windy","temperature":"cold"}
reduced=[r for r in examples if any(r[f]!=heldout[f] for f in features)]
new_tree=id3(reduced,features)
print("Held-out prediction",classify(new_tree,heldout))
print("Original answer","no")